<a href="https://colab.research.google.com/github/0MSA0/supreme-robot/blob/main/data_loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Data-Loader

In [ ]:
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader

class SupremeDataset(Dataset):
  def __init__(self, input_text: str, tokenizer, max_length, stride) -> None:
    self.input_ids = []
    self.target_ids = []

    encoded_words = tokenizer.encode(input_text)

    # Use a sliding window to chunk the input text into overlapping sequences of max_length
    for i in range(0, len(encoded_words) - max_length, stride):
      input_chunk = encoded_words[i:i + max_length]
      target_chunk = encoded_words[i + 1: i + max_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]


In [ ]:
def create_supreme_dataloader(txt, tokenizer, batch_size=4, max_length=256,
                              stride=128, shuffle=True, drop_last=True,
                              num_workers=0):

    # Create dataset
    dataset = SupremeDataset(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:
tokenizer = tiktoken.get_encoding("o200k_base")
vocab_size = tokenizer.n_vocab
output_dim = 256
context_length = 1024

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

batch_size = 8
max_length = 4
dataloader = create_supreme_dataloader(
    raw_text,
    tokenizer,
    batch_size=batch_size,
    max_length=max_length,
    stride=max_length
)

token_embeddings = []
pos_embeddings = []
for batch in dataloader:
    input_token_ids, target_token_ids = batch
    token_embeddings.append(token_embedding_layer(input_token_ids))
    pos_embeddings.append(pos_embedding_layer(torch.arange(max_length)))

print(token_embeddings[0])
print(pos_embeddings[0])

tensor([[[-0.3185, -1.3122, -0.4342,  ...,  0.3373,  0.2529,  0.5604],
         [-0.9526,  1.7524, -1.1171,  ..., -0.0175,  0.6064, -0.5777],
         [-0.2834,  1.0330,  0.4660,  ...,  0.7541, -0.2107, -1.0264],
         [ 1.7017, -1.3565, -0.8629,  ...,  0.1906,  1.4000,  0.3398]],

        [[-1.3920, -0.2044,  1.0056,  ...,  1.1797, -1.1462, -0.9753],
         [-0.3809, -0.6195,  1.5024,  ..., -1.1246,  0.4186,  0.8338],
         [ 0.5049,  0.4613,  0.4244,  ...,  0.0423,  0.8618, -1.0340],
         [-0.1870,  0.5779, -0.4669,  ...,  0.2249,  0.9636, -0.3963]],

        [[ 1.5575, -1.3181, -0.6401,  ...,  1.5962,  1.3634,  0.1956],
         [-0.8558, -1.1146, -0.6281,  ..., -1.1353,  0.9116, -0.4942],
         [-2.4153,  0.7933, -1.9186,  ...,  1.9109,  0.5211,  0.3359],
         [ 0.0995,  1.2996, -0.5328,  ...,  0.1865, -1.2035, -0.6374]],

        ...,

        [[-0.6149, -1.9723, -1.3583,  ..., -0.4161,  1.0515,  0.8601],
         [-2.4153,  0.7933, -1.9186,  ...,  1.9109,  0.52

Some really important stuff:

In [ ]:
print("Hello there")

Hello there


In [ ]:
print("I have the high ground")

I have the high ground
